# 03. Regularization - Ridge, Lasso, Elastic Net**Topic:** Compare unregularized linear regression with Ridge (L2), Lasso (L1), and Elastic Net.**Dataset:** California Housing (sklearn).**What you will do:**1. Train a plain Linear Regression baseline2. Train Ridge, Lasso, Elastic Net with the same data3. Compare RMSE and R²4. Visualize how coefficients shrink as regularization increases5. Use cross-validated tuning to pick the best `alpha`

In [ ]:
# Run this cell first if running locally. In Google Colab everything is pre-installed.# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy xgboost lightgbm

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snssns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)np.random.seed(42)

In [ ]:
from sklearn.datasets import fetch_california_housingfrom sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.preprocessing import StandardScalerfrom sklearn.pipeline import Pipelinefrom sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCVfrom sklearn.metrics import mean_squared_error, r2_scoredata = fetch_california_housing(as_frame=True)df = data.frameX = df.drop(columns=["MedHouseVal"])y = df["MedHouseVal"]X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 1. Compare four models

In [ ]:
def evaluate(name, model):    pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])    pipe.fit(X_train, y_train)    pred = pipe.predict(X_test)    rmse = mean_squared_error(y_test, pred, squared=False)    r2 = r2_score(y_test, pred)    print(f"{name:18s}  RMSE = {rmse:.4f}   R^2 = {r2:.4f}")    return pipemodels = {    "Linear":       LinearRegression(),    "Ridge":        Ridge(alpha=1.0),    "Lasso":        Lasso(alpha=0.1),    "Elastic Net":  ElasticNet(alpha=0.1, l1_ratio=0.5),}fitted = {name: evaluate(name, m) for name, m in models.items()}

## 2. Visualize how coefficients shrink as alpha increases

In [ ]:
alphas = np.logspace(-3, 3, 50)ridge_coefs = []lasso_coefs = []scaler = StandardScaler().fit(X_train)X_train_s = scaler.transform(X_train)for a in alphas:    r = Ridge(alpha=a).fit(X_train_s, y_train)    l = Lasso(alpha=a, max_iter=5000).fit(X_train_s, y_train)    ridge_coefs.append(r.coef_)    lasso_coefs.append(l.coef_)ridge_coefs = np.array(ridge_coefs)lasso_coefs = np.array(lasso_coefs)fig, axes = plt.subplots(1, 2, figsize=(14, 5))for i, name in enumerate(X.columns):    axes[0].plot(alphas, ridge_coefs[:, i], label=name)    axes[1].plot(alphas, lasso_coefs[:, i], label=name)for ax, title in zip(axes, ["Ridge (L2)", "Lasso (L1)"]):    ax.set_xscale("log")    ax.set_xlabel("alpha")    ax.set_ylabel("coefficient")    ax.set_title(title)    ax.legend(fontsize=8)plt.tight_layout()plt.show()

Notice how Lasso drives coefficients exactly to zero, while Ridge shrinks them gently.

## 3. Pick the best alpha with cross validation

In [ ]:
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 50)).fit(X_train_s, y_train)lasso_cv = LassoCV(alphas=np.logspace(-3, 3, 50), cv=5, max_iter=5000).fit(X_train_s, y_train)print(f"Best Ridge alpha: {ridge_cv.alpha_:.5f}")print(f"Best Lasso alpha: {lasso_cv.alpha_:.5f}")

## 4. Exercises1. **Generate noisy fake features** (random columns) and check that Lasso zeroes them out.2. **Try `Lasso(alpha=10)`** - what happens to the model?3. **Plot RMSE vs alpha** for Ridge and Lasso. Where is the sweet spot?4. **Try a real Kaggle dataset** like the Ames Housing dataset (more features = regularization matters more).